# Data Acquisition Pipeline - Google Colab

This notebook automates cloning the repository and running the 54-table data acquisition pipeline using Google Colab. The database will be stored persistently in your Google Drive.

## Step 1: Clone Repository

Run the cell below to clone the repository to your Colab session. If the repository is public, you can leave the token field blank.

In [ ]:
#@title GitHub Authentication (Optional)
#@markdown Enter your GitHub Personal Access Token (PAT) if the repository is private (otherwise leave it blank):
github_token = "" #@param {type:"string"}

if github_token:
    !git clone https://{github_token}@github.com/nithin12342/data-analysis-and-2026-ai-projection.git
else:
    !git clone https://github.com/nithin12342/data-analysis-and-2026-ai-projection.git

%cd data-analysis-and-2026-ai-projection

## Step 2: Upload Relevant Project Files (Optional)

Because folders like `DATA/` (containing SEC DERA source data) and files like `.env` (containing API keys) are typically git-ignored, you can upload them to your Colab session here so the pipeline can use them.

In [ ]:
#@title Upload `.env` file
#@markdown Run this cell to upload your `.env` file containing secrets/keys (e.g. `GITHUB_TOKEN`):
from google.colab import files
import os

uploaded = files.upload()
for fn in uploaded.keys():
    if fn == '.env':
        os.rename('.env', '/content/data-analysis-and-2026-ai-projection/.env')
        print("Successfully uploaded and moved .env file.")

In [ ]:
#@title Mount Google Drive & Copy `DATA` Folder
#@markdown If you have your `DATA` folder (containing SEC quarterly folders) saved on Google Drive, run this cell to mount Drive and copy it to the workspace:
from google.colab import drive
import os

drive.mount('/content/drive')

drive_data_path = '/content/drive/MyDrive/DATA'
if os.path.exists(drive_data_path):
    print("Copying DATA folder from Google Drive...")
    !cp -r /content/drive/MyDrive/DATA /content/data-analysis-and-2026-ai-projection/
    print("DATA folder copied successfully!")
else:
    print("DATA folder not found in Google Drive. You can manually upload SEC files via the Colab files tab or skip if fetching data from web.")

## Step 3: Install Requirements

Install the required libraries (DuckDB, Pandas, requests, etc.) to set up the ingestion environment.

In [ ]:
# Install dependencies
!pip install -r scripts/acquisition/requirements.txt

## Step 4: Run Ingestion Pipeline

Run the orchestrator script. Adding `--mount-drive` will prompt you to connect Google Drive, enabling the script to persistently save the DuckDB database to your Drive root under `master_consolidated.duckdb` so you can retrieve it later.

In [ ]:
# Run all data acquisition clusters (A, B, D, E) and build derived tables
!python scripts/acquisition/colab_pipeline.py --cluster all --mount-drive

## Step 5: Verify Database Ingestion

After ingestion completes, you can run this cell to verify that the tables were created successfully and count the number of rows populated in each.

In [ ]:
import duckdb

# Connect to the database stored on your Google Drive
db_path = "/content/drive/MyDrive/master_consolidated.duckdb"
try:
    con = duckdb.connect(db_path)
    tables = con.execute("SHOW TABLES").fetchall()
    print(f"Database verified successfully. Found {len(tables)} tables:")
    for t in tables:
        count = con.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
        print(f"  - {t[0]}: {count} rows")
    con.close()
except Exception as e:
    print(f"Error opening database at {db_path}: {e}")
    print("Make sure you approved Google Drive access and completed the previous step.")